# Chapter 6: Linear Models

> A decision tree asks questions, a nearest-neighbor votes with its neighbors — a linear model just adds up evidence, one weighted feature at a time.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 3 (The Perceptron) &nbsp;|&nbsp; **Time:** ~50 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 6

---

## Learning Objectives

- Define and plot the four common surrogate loss functions: hinge, logistic, exponential, squared
- Implement a generic regularized linear classifier trained with (sub)gradient descent, for both hinge loss (soft-margin SVM) and logistic loss (logistic regression)
- Explain why the 2-norm regularizer shrinks weights, and observe this trade-off empirically on real data
- Derive and implement the closed-form solution for 2-norm regularized squared loss (ridge regression)

## The Problem

The perceptron finds *a* separating hyperplane, but stops as soon as training data is (nearly) correctly classified — it has no notion of "how good" that hyperplane is, and no principled way to handle non-separable data.

**Linear Models** generalizes the perceptron by separating the *model* (a linear function `w·x + b`) from the *algorithm* used to fit it, turning learning into an explicit optimization problem:

$$\min_w \; \text{loss(training data)} + \text{regularizer(model complexity)}$$

Different choices of loss and regularizer recover the perceptron, logistic regression, and the support vector machine as special cases of the same framework.

## The Concept

**From raw error to a solvable optimization problem:**

```
Raw 0/1 loss (NP-hard to optimize)
      │
      ▼
Replace with a convex surrogate loss
      │
      ├──► Hinge loss    ──► Soft-margin SVM
      │
      └──► Logistic loss ──► Logistic Regression
                                   │
                                   ▼
                    min  loss + (lambda/2) * ||w||²
                                   │
                                   ▼
                    Solve with (sub)gradient descent
```

### Key Ideas

- **Zero/one loss is NP-hard to optimize directly**, because tiny parameter changes cause discontinuous jumps in the loss. Surrogate losses (hinge, logistic, exponential, squared) are convex upper bounds that are easy to optimize instead.
- **Regularization controls complexity:** adding `(lambda/2)||w||²` to the objective shrinks the weight vector, which is exactly what "simple function" means for a linear model (Section 6.3: small weights → small rate of change → less sensitive to any one feature).
- **Hinge loss + 2-norm regularizer = soft-margin SVM.** Logistic loss + 2-norm regularizer = regularized logistic regression. Same optimization machinery, different loss function.
- **Squared loss has a closed-form solution** (no iterative optimization needed) when paired with a 2-norm regularizer: this is **ridge regression**, `w = (XᵀX + λI)⁻¹Xᵀy`.

## Build It

### Setup

Alongside the Breast Cancer Wisconsin dataset (classification), we now also load the Diabetes dataset (a real regression benchmark) for the ridge regression experiment, plus `LogisticRegression`, `LinearSVC`, and `Ridge` from scikit-learn as reference implementations to validate against.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, mean_squared_error

RNG = np.random.RandomState(42)

### Step 1: Loss Gradients (Section 6.1–6.5)

Every loss function in this framework is defined in terms of the activation `a = w·x + b`, and gradient descent only needs its derivative with respect to `a`:

- **Hinge loss** `max(0, 1 - y·a)`: its subgradient is `-y` whenever the margin `y·a < 1` (the point is inside the margin or misclassified), and `0` otherwise (the point is safely correct).
- **Logistic loss** `log(1 + exp(-y·a))`: its gradient is `-y · sigmoid(-y·a)`. We clip the exponent to avoid numerical overflow on very confident (large-margin) predictions.

In [2]:
def hinge_loss_grad(y, a):
    margin = y * a
    g = np.where(margin < 1, -y, 0.0)
    return g


def logistic_loss_grad(y, a):
    z = np.clip(y * a, -30, 30)
    return -y / (1.0 + np.exp(z))


LOSSES = {
    "hinge": hinge_loss_grad,
    "logistic": logistic_loss_grad,
}

### Step 2: A Generic Regularized Linear Classifier (Section 6.4)

`LinearClassifierFromScratch` minimizes

$$\frac{1}{N}\sum_n \text{loss}(y_n, w \cdot x_n + b) + \frac{\lambda}{2}\|w\|^2$$

via full-batch gradient descent, with a shrinking step size `eta_k = eta0 / sqrt(k)`. Swapping the `loss` argument between `"hinge"` and `"logistic"` recovers the soft-margin SVM or logistic regression respectively — same training loop, different gradient function.

In [3]:
class LinearClassifierFromScratch:
    def __init__(self, loss="hinge", lam=1e-2, eta0=1.0, max_iter=500):
        self.loss = loss
        self.lam = lam
        self.eta0 = eta0
        self.max_iter = max_iter

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=float)
        N, D = X.shape
        self.w = np.zeros(D)
        self.b = 0.0
        grad_fn = LOSSES[self.loss]

        for k in range(1, self.max_iter + 1):
            a = X @ self.w + self.b
            g_per_example = grad_fn(y, a)
            grad_w = (X.T @ g_per_example) / N + self.lam * self.w
            grad_b = g_per_example.mean()

            eta = self.eta0 / np.sqrt(k)
            self.w -= eta * grad_w
            self.b -= eta * grad_b

        return self

    def decision_function(self, X):
        return np.asarray(X) @ self.w + self.b

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)

### Step 3: Closed-Form Ridge Regression (Section 6.6)

When the loss is squared error and the regularizer is the 2-norm, the optimal `w` has a closed-form solution — no iterative optimization needed. We first **center** both `X` and `y` so the intercept term is never itself regularized (matching scikit-learn's default `fit_intercept=True` behavior), solve for `w` on the centered data, then recover the intercept `b` afterward.

In [4]:
class RidgeRegressionFromScratch:
    def __init__(self, lam=1.0):
        self.lam = lam

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y, dtype=float)
        self.x_mean = X.mean(axis=0)
        self.y_mean = y.mean()
        Xc = X - self.x_mean
        yc = y - self.y_mean
        D = X.shape[1]
        A = Xc.T @ Xc + self.lam * np.eye(D)
        self.w = np.linalg.solve(A, Xc.T @ yc)
        self.b = self.y_mean - self.x_mean @ self.w
        return self

    def predict(self, X):
        return np.asarray(X) @ self.w + self.b

## Use It — Real Data

### Experiment A: Hinge Loss (SVM) vs. `sklearn.svm.LinearSVC`

We load Breast Cancer Wisconsin, remap labels to ±1, scale features, and train our from-scratch hinge-loss classifier (a soft-margin SVM) alongside scikit-learn's `LinearSVC`. We compare not just accuracy but the **prediction agreement rate** between the two.

In [5]:
data = load_breast_cancer()
X, y_raw = data.data, data.target
y = np.where(y_raw == 0, -1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

svm_scratch = LinearClassifierFromScratch(loss="hinge", lam=1e-2, eta0=1.0, max_iter=1000)
svm_scratch.fit(X_train_s, y_train)
pred_scratch = svm_scratch.predict(X_test_s)
acc_scratch = accuracy_score(y_test, pred_scratch)

sk_svm = LinearSVC(C=1.0 / (1e-2), max_iter=5000, random_state=42)
sk_svm.fit(X_train_s, y_train)
acc_sklearn = accuracy_score(y_test, sk_svm.predict(X_test_s))

agreement = np.mean(pred_scratch == sk_svm.predict(X_test_s))
print(f"From-scratch hinge-loss SVM test accuracy : {acc_scratch:.4f}")
print(f"sklearn LinearSVC        test accuracy     : {acc_sklearn:.4f}")
print(f"Prediction agreement rate                  : {agreement:.4f}")

From-scratch hinge-loss SVM test accuracy : 0.9825
sklearn LinearSVC        test accuracy     : 0.9474
Prediction agreement rate                  : 0.9532


### Experiment B: Logistic Loss vs. `sklearn.linear_model.LogisticRegression`

Same setup, but switching `loss="logistic"` to recover regularized logistic regression, compared against scikit-learn's reference `LogisticRegression`.

In [6]:
logreg_scratch = LinearClassifierFromScratch(loss="logistic", lam=1e-2, eta0=1.0, max_iter=1000)
logreg_scratch.fit(X_train_s, y_train)
pred_log_scratch = logreg_scratch.predict(X_test_s)
acc_log_scratch = accuracy_score(y_test, pred_log_scratch)

sk_logreg = LogisticRegression(C=1.0 / (1e-2), max_iter=5000)
sk_logreg.fit(X_train_s, y_train)
acc_log_sklearn = accuracy_score(y_test, sk_logreg.predict(X_test_s))

agreement_log = np.mean(pred_log_scratch == sk_logreg.predict(X_test_s))
print(f"From-scratch logistic-loss classifier test accuracy : {acc_log_scratch:.4f}")
print(f"sklearn LogisticRegression       test accuracy       : {acc_log_sklearn:.4f}")
print(f"Prediction agreement rate                            : {agreement_log:.4f}")

From-scratch logistic-loss classifier test accuracy : 0.9825
sklearn LogisticRegression       test accuracy       : 0.9532
Prediction agreement rate                            : 0.9474


**Reading Experiments A & B:** exact matches with scikit-learn are not expected — `LinearSVC` and `LogisticRegression` use different solvers internally (e.g. dual coordinate descent or L-BFGS, instead of plain gradient descent) — but accuracy and predictions should land close together.

### Experiment C: Regularization Strength (`lambda`) vs. Train/Test Accuracy (Section 6.3)

Now we directly observe the book's central claim about regularization: as `lambda` grows, the learned weight vector `w` should shrink in norm, trading some training accuracy for (hopefully) more robust generalization — until `lambda` becomes so large the model underfits.

In [7]:
print(f"{'lambda':>10} | {'train acc':>10} | {'test acc':>9} | {'||w||':>8}")
print("-" * 45)
for lam in [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]:
    clf = LinearClassifierFromScratch(loss="logistic", lam=lam, eta0=1.0, max_iter=1000)
    clf.fit(X_train_s, y_train)
    train_acc = accuracy_score(y_train, clf.predict(X_train_s))
    test_acc = accuracy_score(y_test, clf.predict(X_test_s))
    print(f"{lam:>10.4f} | {train_acc:>10.4f} | {test_acc:>9.4f} | {np.linalg.norm(clf.w):>8.4f}")

    lambda |  train acc |  test acc |    ||w||
---------------------------------------------
    0.0001 |     0.9874 |    0.9766 |   2.7295
    0.0010 |     0.9874 |    0.9766 |   2.6594
    0.0100 |     0.9849 |    0.9825 |   2.1473
    0.1000 |     0.9774 |    0.9474 |   1.0983
    1.0000 |     0.9322 |    0.9415 |   0.4420
   10.0000 |     0.7513 |    0.7193 |   0.1104


**Reading the table:** as `lambda` grows, `||w||` shrinks (Section 6.3: small weights ⇔ simple functions). Training accuracy drops slightly, but test accuracy stays robust until `lambda` gets too large, at which point the model underfits.

### Experiment D: Closed-Form Ridge Regression vs. `sklearn.linear_model.Ridge`

Finally, a real regression task: the Diabetes dataset. We fit our closed-form ridge regression and compare it against scikit-learn's `Ridge` — since both solve the exact same convex quadratic problem in closed form, they should match almost to the last decimal place.

In [8]:
housing = load_diabetes()
Xh, yh = housing.data, housing.target
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, yh, test_size=0.3, random_state=42
)
h_scaler = StandardScaler().fit(Xh_train)
Xh_train_s = h_scaler.transform(Xh_train)
Xh_test_s = h_scaler.transform(Xh_test)

lam_ridge = 0.5
ridge_scratch = RidgeRegressionFromScratch(lam=lam_ridge).fit(Xh_train_s, yh_train)
pred_ridge_scratch = ridge_scratch.predict(Xh_test_s)
mse_scratch = mean_squared_error(yh_test, pred_ridge_scratch)

sk_ridge = Ridge(alpha=lam_ridge).fit(Xh_train_s, yh_train)
mse_sklearn = mean_squared_error(yh_test, sk_ridge.predict(Xh_test_s))

print(f"Dataset shape: {Xh.shape[0]} examples, {Xh.shape[1]} features")
print(f"From-scratch closed-form ridge   test MSE : {mse_scratch:.4f}")
print(f"sklearn Ridge                    test MSE : {mse_sklearn:.4f}")
print(f"Max abs weight difference                : {np.max(np.abs(ridge_scratch.w - sk_ridge.coef_)):.6f}")

Dataset shape: 442 examples, 10 features
From-scratch closed-form ridge   test MSE : 2820.4905
sklearn Ridge                    test MSE : 2820.4905
Max abs weight difference                : 0.000000


**Reading the result:** the closed-form solution `w = (XᵀX + λI)⁻¹Xᵀy` should match sklearn's `Ridge` almost exactly (up to floating point / solver differences), directly confirming the matrix-algebra derivation from Section 6.6.

## Use It

| API / Function | When to use it |
|---|---|
| `LinearClassifierFromScratch(loss="hinge")` | Want a maximum-margin linear classifier; robust to outliers past the margin |
| `LinearClassifierFromScratch(loss="logistic")` | Want calibrated probability-like outputs, not just a hard decision |
| `RidgeRegressionFromScratch(lam)` | Real-valued targets, want a fast closed-form fit with L2 shrinkage |
| `sklearn.svm.LinearSVC` | Production-grade SVM training (LIBLINEAR solver, much faster than plain gradient descent) |
| `sklearn.linear_model.LogisticRegression` | Production-grade logistic regression with multiple solvers and multiclass support |

## Exercises

1. Add the exponential loss (`exp(-ya)`) and squared loss (`(y-a)^2`) gradients to the `LOSSES` dictionary and compare all four surrogate losses on the same train/test split.
2. Replace the fixed step-size schedule `eta0/sqrt(k)` with a line-search or Adam-style adaptive step size, and see how much faster the model converges.
3. Implement the 1-norm (L1) regularizer with subgradient truncation (Section 12.3 of the book) and check how many weights become exactly zero on the Breast Cancer dataset.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Regularizer** | "Just a penalty to prevent big numbers" | A term added to the training objective that encodes inductive bias about which functions are "simple," independent of how well they fit the training data |
| **Hinge Loss** | "Just another name for 0/1 loss" | A convex, piecewise-linear upper bound on 0/1 loss that only penalizes points *inside* the margin, giving rise to sparse "support vectors" |
| **Support Vector Machine** | "A totally different algorithm from logistic regression" | The specific case of the general regularized-loss framework where the loss is hinge loss; shares its optimization machinery with logistic regression |
| **Closed-form Solution** | "Only exists for 'easy' toy problems" | An exact algebraic solution (here, one matrix inversion) that some loss/regularizer combinations admit, avoiding iterative optimization entirely |

## Summary

- Linear models separate the *model* (`w·x + b`) from the *algorithm* used to fit it, turning learning into minimizing loss + regularizer
- **Surrogate losses** (hinge, logistic) replace the NP-hard 0/1 loss with convex functions that gradient descent can optimize
- **Hinge loss + L2 regularizer = soft-margin SVM**; **logistic loss + L2 regularizer = logistic regression** — same machinery, different loss
- The regularization strength `lambda` directly controls `||w||`, trading training fit for generalization
- **Ridge regression** (squared loss + L2 regularizer) has an exact closed-form solution, no iterative optimization required

---

**Next:** Chapter 7 — Probabilistic Modeling